# 🍼 Infant State Recognition System — Phase 1
**AML Project 3 | Domain: Healthcare, Assistive Technology, Consumer IoT**

---

**Goal:** Listen to a baby cry and identify what the baby needs.

**Five states:** `hungry` · `pain` · `discomfort` · `burping` · `tired`

**Phase 1 covers:**
- ✅ Baseline ML — Decision Tree (interpretable)
- ✅ Advanced ML — Random Forest + SVM with GridSearchCV

> Phase 2 will add: Deep Learning (CNN) and Edge/Hybrid model

---

## Step 1: Install & Import Everything

In [ ]:
!pip install librosa soundfile -q

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import warnings
import os
import joblib

from pathlib import Path
from joblib import Parallel, delayed
from scipy import stats

# scikit-learn
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      GridSearchCV, cross_val_score)
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import f_classif

warnings.filterwarnings("ignore")
os.makedirs("data/raw", exist_ok=True)
os.makedirs("models",   exist_ok=True)
os.makedirs("reports",  exist_ok=True)

LABELS = ["hungry", "pain", "discomfort", "burping", "tired"]
COLORS = ["#2196F3", "#F44336", "#FF9800", "#4CAF50", "#9C27B0"]
RANDOM_STATE = 42

print("✅ All imports successful")

## Step 2: Create the Dataset

We generate 1,000 synthetic audio clips (200 per class) that mimic real infant cry acoustics.

Each class has distinct acoustic properties:

| Class | Pitch | Rhythm | Harshness | What it sounds like |
|---|---|---|---|---|
| Hungry | 400 Hz | 1.2 Hz | Low | Rhythmic, medium pitched |
| Pain | 600 Hz | 0.5 Hz | Very high | High pitched, urgent, harsh |
| Discomfort | 350 Hz | 0.9 Hz | Medium | Steady, moderate |
| Burping | 250 Hz | 2.0 Hz | Very low | Fast, clean, low pitched |
| Tired | 300 Hz | 1.5 Hz | Low | Gentle, slow |

> **Using real data?** Replace `data/raw/` with the [donateacry-corpus](https://github.com/gveres/donateacry-corpus) folders and skip this cell.

In [ ]:
# Acoustic parameters for each cry type
CRY_PARAMS = {
    "hungry":     {"pitch": 400, "pitch_spread": 60,  "rhythm": 1.2, "noise": 0.3},
    "pain":       {"pitch": 600, "pitch_spread": 200, "rhythm": 0.5, "noise": 0.8},
    "discomfort": {"pitch": 350, "pitch_spread": 80,  "rhythm": 0.9, "noise": 0.5},
    "burping":    {"pitch": 250, "pitch_spread": 40,  "rhythm": 2.0, "noise": 0.2},
    "tired":      {"pitch": 300, "pitch_spread": 50,  "rhythm": 1.5, "noise": 0.4},
}

SAMPLE_RATE = 22050
DURATION    = 3.0

def make_one_cry(label, seed):
    """Generates one synthetic cry audio clip."""
    rng    = np.random.default_rng(seed)
    params = CRY_PARAMS[label]
    t      = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION))

    # Fundamental frequency + harmonics (gives the cry its 'voice')
    pitch  = np.clip(params["pitch"] + rng.normal(0, params["pitch_spread"]), 100, 1200)
    signal = np.sin(2 * np.pi * pitch * t)
    for h in range(2, 6):
        signal += (1 / h) * np.sin(2 * np.pi * pitch * h * t)

    # Amplitude modulation — the cry pulses rhythmically
    rhythm = params["rhythm"] + rng.normal(0, 0.1)
    signal = signal * (0.5 + 0.5 * np.sin(2 * np.pi * rhythm * t))

    # Add broadband noise (harshness)
    signal = signal + rng.normal(0, params["noise"], len(t))

    # Normalise to [-1, 1]
    return (signal / (np.max(np.abs(signal)) + 1e-8)).astype(np.float32)


def create_dataset(samples_per_class=200):
    seed = 0
    for label in LABELS:
        Path(f"data/raw/{label}").mkdir(parents=True, exist_ok=True)
        for i in range(samples_per_class):
            sf.write(f"data/raw/{label}/{i:04d}.wav",
                     make_one_cry(label, seed), SAMPLE_RATE)
            seed += 1
    print(f"✅ Created {samples_per_class * len(LABELS)} audio clips")


create_dataset(samples_per_class=200)

## Step 3: Feature Extraction (88 dimensions)

Raw audio = 66,150 numbers per clip. We can't feed that to ML models directly.
Instead we extract **88 meaningful features** that describe each clip compactly.

### Why 8 kHz instead of 22 kHz?
Infant cry fundamentals are 300–800 Hz.
The **Nyquist theorem** says: sample rate must be > 2 × max frequency of interest.
So 8 kHz (Nyquist = 4 kHz) captures everything we need at **2.75× less compute**.

### What are MFCCs?
Mel-Frequency Cepstral Coefficients describe the "shape" of the sound spectrum at each moment.
They're the most important feature in audio/speech ML — a compact fingerprint of the vocal tract.

### Our 88 features:
| Group | Dims | What it captures |
|---|---|---|
| MFCC mean + std (20 coeffs) | 40 | Vocal tract shape |
| MFCC Delta (rate of change) | 20 | Temporal dynamics |
| MFCC Delta-Delta (acceleration) | 20 | Pulsed vs sustained cries |
| Spectral centroid, flatness, ZCR, RMS | 8 | Brightness, roughness, loudness |

In [ ]:
SR     = 8000   # 8 kHz — enough for infant cries
N_MFCC = 20
N_FFT  = 512
HOP    = 128

def extract_features(file_path):
    """
    Reads one WAV file and returns an 88-number feature vector.
    """
    # Load at 8 kHz, exactly 3 seconds
    audio, _ = librosa.load(file_path, sr=SR, duration=3.0, mono=True)
    target = int(SR * 3.0)
    audio = np.pad(audio, (0, max(0, target - len(audio))))[:target]

    # Pre-emphasis: slightly boosts high frequencies
    audio_pre = librosa.effects.preemphasis(audio)

    # ── MFCC features ──────────────────────────────────────────────────────
    mfcc   = librosa.feature.mfcc(y=audio_pre, sr=SR, n_mfcc=N_MFCC,
                                   n_fft=N_FFT, hop_length=HOP)
    delta1 = librosa.feature.delta(mfcc)           # rate of change
    delta2 = librosa.feature.delta(mfcc, order=2)  # acceleration

    # ── Other spectral features ─────────────────────────────────────────────
    zcr      = librosa.feature.zero_crossing_rate(audio, hop_length=HOP)[0]
    rms      = librosa.feature.rms(y=audio, hop_length=HOP)[0]
    mel      = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=32,
                                               n_fft=N_FFT, hop_length=HOP)
    centroid = librosa.feature.spectral_centroid(y=audio, sr=SR,
                                                  n_fft=N_FFT, hop_length=HOP)[0]
    mel_db   = librosa.power_to_db(mel)

    # ── Assemble the 88-number vector ───────────────────────────────────────
    features = []
    features.extend(np.mean(mfcc,   axis=1))    # 20 — average shape
    features.extend(np.std(mfcc,    axis=1))    # 20 — variation in shape
    features.extend(np.mean(delta1, axis=1))    # 20 — rate of change
    features.extend(np.mean(delta2, axis=1))    # 20 — acceleration
    features.extend([np.mean(zcr),  np.std(zcr)])       # 2 — roughness
    features.extend([np.mean(rms),  np.std(rms)])       # 2 — loudness
    features.extend([np.mean(mel_db), np.std(mel_db)])  # 2 — spectral energy
    features.extend([np.mean(centroid), np.std(centroid)])  # 2 — brightness
    return np.array(features, dtype=np.float32)  # 88 total


# ── Collect file paths and labels ───────────────────────────────────────────
all_paths, all_labels = [], []
for label_idx, label in enumerate(LABELS):
    wavs = sorted(Path("data/raw").glob(f"{label}/*.wav"))
    all_paths  += [str(w) for w in wavs]
    all_labels += [label_idx] * len(wavs)

# ── Extract in parallel (faster) ────────────────────────────────────────────
print(f"Extracting features from {len(all_paths)} files...")
results = Parallel(n_jobs=-1)(delayed(extract_features)(p) for p in all_paths)

X = np.array(results, dtype=np.float32)   # (1000, 88)
y = np.array(all_labels, dtype=np.int32)  # (1000,)

print(f"✅ Feature matrix: {X.shape}")
print(f"   Each row = 1 audio clip | Each column = 1 feature")

## Step 4: Exploratory Data Analysis (EDA)

Before training any model, we explore the data to understand it.
This is not just required — it actively informs our modelling decisions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("EDA — Phase 1 Overview", fontsize=14, fontweight="bold")

# 1. Class balance
counts = np.bincount(y)
axes[0].bar(LABELS, counts, color=COLORS, edgecolor="black")
for i, c in enumerate(counts):
    axes[0].text(i, c + 2, str(c), ha="center", fontweight="bold")
axes[0].set_title(f"Class Distribution (imbalance: {counts.max()/counts.min():.2f}x)")
axes[0].set_ylabel("Number of clips")

# 2. PCA — do our features separate the classes?
X_scaled = StandardScaler().fit_transform(X)
pca      = PCA(n_components=2, random_state=42).fit(X_scaled)
X_pca    = pca.transform(X_scaled)
for i, (lbl, col) in enumerate(zip(LABELS, COLORS)):
    mask = (y == i)
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=col, label=lbl, alpha=0.6, s=15)
axes[1].set_title(f"PCA — Feature Space\nVariance explained: {pca.explained_variance_ratio_.sum():.1%}")
axes[1].legend(fontsize=7)

# 3. ANOVA F-scores — which features are most discriminative?
F_scores, _ = f_classif(X, y)
top10 = np.argsort(F_scores)[-10:][::-1]
axes[2].barh(range(10), F_scores[top10][::-1], color="#FF9800")
axes[2].set_yticks(range(10))
axes[2].set_yticklabels([f"Feature {i}" for i in top10[::-1]], fontsize=8)
axes[2].set_title("Top 10 Most Discriminative Features\n(ANOVA F-score)")
axes[2].set_xlabel("F-score")

plt.tight_layout()
plt.savefig("reports/eda_overview.png", dpi=150)
plt.show()
print("Finding: PCA shows clear clusters → features are already discriminative before training")

In [ ]:
# Waveform + Mel-spectrogram per class
fig, axes = plt.subplots(5, 2, figsize=(14, 15))
fig.suptitle("How each cry type looks — Waveform (left) & Mel-Spectrogram (right)",
             fontsize=13, fontweight="bold")

for i, label in enumerate(LABELS):
    path = sorted(Path("data/raw").glob(f"{label}/*.wav"))[0]
    audio, sr_orig = librosa.load(str(path), sr=22050, duration=3.0)
    t = np.linspace(0, len(audio) / sr_orig, len(audio))

    # Waveform
    axes[i, 0].plot(t, audio, color=COLORS[i], linewidth=0.5)
    axes[i, 0].set_ylabel(f"{label}", fontweight="bold", fontsize=11)
    if i == 4: axes[i, 0].set_xlabel("Time (s)")
    if i == 0: axes[i, 0].set_title("Waveform")

    # Mel-spectrogram
    mel = librosa.feature.melspectrogram(y=audio, sr=sr_orig, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr_orig, x_axis="time",
                                    y_axis="mel", ax=axes[i, 1], cmap="magma")
    plt.colorbar(img, ax=axes[i, 1], format="%+2.0f dB")
    if i == 0: axes[i, 1].set_title("Mel-Spectrogram")

plt.tight_layout()
plt.savefig("reports/spectrograms.png", dpi=150)
plt.show()
print("Notice: Pain has the highest pitch and brightest spectrogram. Burping is the cleanest.")

In [ ]:
# t-SNE — better than PCA for revealing cluster structure
print("Running t-SNE (takes ~30 seconds)...")
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30,
              max_iter=1000).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for i, (lbl, col) in enumerate(zip(LABELS, COLORS)):
    mask = (y == i)
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=col, label=lbl, alpha=0.7, s=25)
ax.set_title("t-SNE: Each dot = one audio clip\n"
             "Clear separate clusters = our 88 features work well!", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("reports/tsne.png", dpi=150)
plt.show()

## Step 5: Train / Test Split

We hold out 20% of data for final testing.
**Stratified split** ensures each class has the same proportion in both sets.
We never use the test set during training — it simulates unseen real-world data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y        # keeps class ratios balanced
)

print(f"Training set : {X_train.shape[0]} clips | {np.bincount(y_train)}")
print(f"Test set     : {X_test.shape[0]} clips | {np.bincount(y_test)}")

## Step 6: Baseline ML — Decision Tree

A Decision Tree asks a series of yes/no questions about the features.

**Example:** "Is MFCC_1 > 5.2? → If yes, probably pain. If no, check the next feature..."

**Why this first?**
- Fully interpretable — you can print and read every single rule
- In clinical settings, nurses need to know *why* the system says "pain"
- Good starting point to check the problem is solvable at all

**Key settings explained:**
- `max_depth=5` → at most 5 questions deep. Prevents memorising training data (overfitting)
- `criterion="gini"` → splits using Gini impurity: G = 1 - Σ p²ᵢ
- `class_weight="balanced"` → treats all classes equally: wᵢ = N / (K × Nᵢ)

In [ ]:
# Pipeline: scale features → train tree
# (Scaling not strictly needed for trees but is good practice)
dt_model = Pipeline([
    ("scaler", StandardScaler()),
    ("tree",   DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ))
])

# ── 5-fold cross-validation ──────────────────────────────────────────────────
# Trains on 5 different data splits and averages the result.
# Gives a more reliable estimate than a single train/test split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(dt_model, X_train, y_train,
                             cv=cv, scoring="f1_macro")
print(f"5-Fold CV F1-macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per fold: {cv_scores.round(3)}")

# ── Train on full training set ───────────────────────────────────────────────
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_f1  = f1_score(y_test, y_pred_dt, average="macro")

print(f"\nTest Accuracy : {dt_acc:.4f}")
print(f"Test Macro F1 : {dt_f1:.4f}")
print(f"\nDetailed classification report:")
print(classification_report(y_test, y_pred_dt, target_names=LABELS))
joblib.dump(dt_model, "models/decision_tree.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Confusion matrix
# Perfect model = all numbers on the diagonal
cm = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title(f"Decision Tree — Confusion Matrix\nAccuracy: {dt_acc:.1%}", fontsize=12)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# Decision tree structure — you can read the actual rules
plot_tree(dt_model["tree"], max_depth=3,
          feature_names=[f"f{i}" for i in range(X.shape[1])],
          class_names=LABELS, filled=True, rounded=True,
          ax=axes[1], fontsize=6)
axes[1].set_title("Decision Tree Rules (top 3 levels shown)", fontsize=12)

plt.tight_layout()
plt.savefig("reports/decision_tree.png", dpi=150)
plt.show()

## Step 7a: Advanced ML — Random Forest

**What is a Random Forest?**
It trains many decision trees on different random subsets of the data, then majority-votes.

**Why is it better than one tree?**
- Single tree: can overfit — memorises specific training examples (high variance)
- 100 trees: each sees different data → errors cancel out → much more stable

**The math:** Ensemble variance ≈ σ²/T where T = number of trees. More trees = lower variance.

**GridSearchCV** tries every combination of these settings and picks the best:
- `n_estimators` → how many trees
- `max_depth` → how deep each tree goes
- `max_features` → how many features each split considers

In [ ]:
rf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("rf",     RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

param_grid_rf = {
    "rf__n_estimators":     [100, 200],
    "rf__max_depth":        [None, 15, 25],
    "rf__min_samples_leaf": [1, 3],
    "rf__max_features":     ["sqrt", "log2"],
}

print("Searching best settings (2-3 minutes)...")
gs_rf = GridSearchCV(
    rf_model, param_grid_rf,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="f1_macro", n_jobs=-1
)
gs_rf.fit(X_train, y_train)

print(f"Best settings : {gs_rf.best_params_}")
print(f"Best CV F1    : {gs_rf.best_score_:.4f}")

y_pred_rf = gs_rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average="macro")

print(f"\nTest Accuracy : {rf_acc:.4f}")
print(f"Test Macro F1 : {rf_f1:.4f}")
print(classification_report(y_test, y_pred_rf, target_names=LABELS))
joblib.dump(gs_rf.best_estimator_, "models/random_forest.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title(f"Random Forest — Confusion Matrix\nAccuracy: {rf_acc:.1%}")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# Feature importance — which features does the forest rely on?
importances = gs_rf.best_estimator_["rf"].feature_importances_
top20 = np.argsort(importances)[-20:]
axes[1].barh(range(20), importances[top20], color="#2196F3")
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f"Feature {i}" for i in top20], fontsize=8)
axes[1].set_xlabel("Gini Importance")
axes[1].set_title("Random Forest — Top 20 Feature Importances")

plt.tight_layout()
plt.savefig("reports/random_forest.png", dpi=150)
plt.show()

## Step 7b: Advanced ML — SVM with RBF Kernel

**What is an SVM?**
It finds the boundary between classes with the **maximum margin** — the widest possible gap.

**What does the RBF kernel do?**
The formula `K(x, y) = exp(-γ × ||x - y||²)` measures similarity between two audio clips.
This effectively maps our 88 features into a much higher-dimensional space where the classes
can be separated by a straight line — even if they can't be in the original 88 dimensions.

**Two settings to tune:**
- `C` → regularisation: high C = stricter separation (risk of overfitting)
- `gamma` → how local the similarity measure is

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),   # SVM requires scaled features — very important!
    ("svm",    SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ))
])

param_grid_svm = {
    "svm__C":     [0.1, 1, 10],
    "svm__gamma": ["scale", "auto", 0.01],
}

print("Searching best SVM settings...")
gs_svm = GridSearchCV(
    svm_model, param_grid_svm,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="f1_macro", n_jobs=-1
)
gs_svm.fit(X_train, y_train)

print(f"Best settings : {gs_svm.best_params_}")
print(f"Best CV F1    : {gs_svm.best_score_:.4f}")

y_pred_svm = gs_svm.predict(X_test)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1  = f1_score(y_test, y_pred_svm, average="macro")

print(f"\nTest Accuracy : {svm_acc:.4f}")
print(f"Test Macro F1 : {svm_f1:.4f}")
print(classification_report(y_test, y_pred_svm, target_names=LABELS))
joblib.dump(gs_svm.best_estimator_, "models/svm.pkl")

In [ ]:
# SVM Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm_svm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt="d", cmap="Purples",
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title(f"SVM (RBF) — Confusion Matrix\nAccuracy: {svm_acc:.1%}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("reports/svm_confusion.png", dpi=150)
plt.show()

## Step 8: Phase 1 Results & Comparison

Comparing all three Phase 1 models side by side.

In [ ]:
# Summary table
results = {
    "Decision Tree (Baseline)": {"acc": dt_acc,  "f1": dt_f1},
    "Random Forest (Advanced)": {"acc": rf_acc,  "f1": rf_f1},
    "SVM RBF (Advanced)":       {"acc": svm_acc, "f1": svm_f1},
}

print(f"{'Model':<30} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 52)
for name, r in results.items():
    best = " ← best" if r["f1"] == max(v["f1"] for v in results.values()) else ""
    print(f"{name:<30} {r['acc']:>10.1%} {r['f1']:>10.4f}{best}")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))

names = list(results.keys())
accs  = [results[m]["acc"] for m in names]
f1s   = [results[m]["f1"]  for m in names]
x     = np.arange(len(names))
w     = 0.35

bars_a = ax.bar(x - w/2, accs, w, label="Accuracy", color="#2196F3", alpha=0.85)
bars_f = ax.bar(x + w/2, f1s,  w, label="Macro F1",  color="#4CAF50", alpha=0.85)

for bar, v in list(zip(bars_a, accs)) + list(zip(bars_f, f1s)):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
            f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")

ax.axhline(0.2, color="red", linestyle="--", alpha=0.5, label="Random chance (5 classes)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Phase 1 Results — Baseline & Advanced ML", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("reports/phase1_comparison.png", dpi=150)
plt.show()

print("\nPhase 1 complete!")
print("Phase 2 will add: Deep Learning (MelCNN) and Edge/Hybrid model")

## Download Results

In [ ]:
import shutil
shutil.make_archive("phase1_reports", "zip", "reports")
shutil.make_archive("phase1_models",  "zip", "models")

from google.colab import files
files.download("phase1_reports.zip")
files.download("phase1_models.zip")
print("Done! Check your downloads folder.")